# Bottom Detector Comparison

This notebook compares the existing CRIMAC-style zarr detectors with several offline labeling-oriented baselines.

Included methods:
- `simple`
- `edge`
- `angle`
- `combined`
- `dp_offline`
- `gradient` (threshold-free NumPy baseline)
- `ensemble_pp` (simple + edge + gradient, then temporal post-processing)
- `seam` (offline shortest-path / seam-style tracker over a cost image)

The notebook focuses on overlays, detector agreement, DP confidence, and mask-level comparisons.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import bottom_detection as np_bottom_detection
from rsi_bottomdetection import (
    angle_bottom_detector,
    combined_bottom_detector,
    dp_offline_bottom_detector,
    edge_bottom_detector,
    simple_bottom_detector,
)
from rsi_bottomdetection.parameters import Parameters

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['image.cmap'] = 'viridis'

In [ ]:
ZARR_PATH = Path.home() / 'Downloads' / 'crimac_output/test.zarr'
PING_SLICE = slice(None)
MAX_RANGE_M = 250.0
FREQUENCY_INDEX = 0

params = Parameters(
    minimum_range=10.0,
    offset=0.5,
    threshold_log_sv=-31.0,
    dp_candidate_threshold_log_sv=-36.0,
    dp_max_candidates_per_ping=6,
    dp_stack_radius=3,
    dp_gradient_weight=1.5,
    dp_energy_weight=1.0,
    dp_contrast_weight=1.0,
    dp_slope_penalty=0.15,
    dp_curvature_penalty=0.08,
    dp_gap_penalty=2.5,
    dp_min_confidence=0.35,
    dp_mask_pad_samples=0,
)

assert ZARR_PATH.exists(), f'Zarr path not found: {ZARR_PATH}'

In [ ]:
def load_dataset(zarr_path, ping_slice=slice(None), max_range_m=None):
    ds = xr.open_zarr(zarr_path, consolidated=False)
    ds = ds.isel(ping_time=ping_slice)
    if max_range_m is not None:
        ds = ds.sel(range=slice(0, max_range_m))
    return ds


def sv_db_from_dataset(ds, frequency_index=0):
    sv_linear = ds.sv.isel(frequency=frequency_index).transpose('ping_time', 'range').values
    return np.where(sv_linear > 0, 10.0 * np.log10(sv_linear), np.nan).T


def heave_draft(ds, frequency_index=0):
    draft = ds.transducer_draft
    if 'frequency' in draft.dims:
        draft = draft.isel(frequency=frequency_index)
    draft = draft.squeeze()
    return (ds.heave + draft).values.astype(np.float64)


def depth_to_range(depth_values, heave_draft_values, offset):
    return depth_values - heave_draft_values + offset


def range_to_depth(range_values, heave_draft_values, offset):
    return range_values + heave_draft_values - offset


def line_to_mask(range_line, range_axis):
    mask = np.zeros((len(range_axis), len(range_line)), dtype=np.uint8)
    bottom_idx = np.full(len(range_line), -1, dtype=np.int64)
    valid = np.isfinite(range_line)
    if valid.any():
        idx = np.searchsorted(range_axis, range_line[valid], side='left')
        idx = np.clip(idx, 0, len(range_axis) - 1)
        bottom_idx[valid] = idx
        valid_cols = np.flatnonzero(valid)
        for col, row in zip(valid_cols, idx):
            mask[row:, col] = 1
    return mask, bottom_idx


def detector_stats(name, depth_values):
    valid = np.isfinite(depth_values)
    diffs = np.abs(np.diff(depth_values[valid])) if valid.sum() > 1 else np.array([np.nan])
    return {
        'detector': name,
        'valid_frac': float(valid.mean()),
        'median_depth_m': float(np.nanmedian(depth_values)) if valid.any() else np.nan,
        'depth_std_m': float(np.nanstd(depth_values)) if valid.any() else np.nan,
        'median_ping_to_ping_absdiff_m': float(np.nanmedian(diffs)),
    }


def compare_to_reference(name, depth_values, reference_values):
    both = np.isfinite(depth_values) & np.isfinite(reference_values)
    delta = depth_values[both] - reference_values[both]
    return {
        'detector': name,
        'n_overlap': int(both.sum()),
        'median_delta_m': float(np.nanmedian(delta)) if both.any() else np.nan,
        'mad_delta_m': float(np.nanmedian(np.abs(delta - np.nanmedian(delta)))) if both.any() else np.nan,
        'rmse_delta_m': float(np.sqrt(np.nanmean(delta ** 2))) if both.any() else np.nan,
    }


def detect_bottom_seam(sv_db, range_axis, minimum_range=10.0, transition_penalty=0.8, upward_penalty=0.8):
    sample_dist = float(range_axis[1] - range_axis[0])
    min_index = max(0, int(minimum_range / sample_dist))
    filled = np.where(np.isnan(sv_db), np.nanmedian(sv_db), sv_db)

    smooth = filled.copy()
    if smooth.shape[0] >= 5:
        kernel = np.ones(5, dtype=np.float64) / 5.0
        smooth = np.apply_along_axis(lambda x: np.convolve(x, kernel, mode='same'), 0, smooth)

    grad = np.gradient(smooth, axis=0)
    support = np.apply_along_axis(lambda x: np.convolve(x, np.ones(7) / 7.0, mode='same'), 0, filled)
    contrast = support - np.roll(support, 6, axis=0)
    cost = -(1.8 * grad + 0.25 * contrast)
    cost[:min_index, :] = np.inf

    n_range, n_pings = cost.shape
    dp = np.full((n_range, n_pings), np.inf, dtype=np.float64)
    back = np.full((n_range, n_pings), -1, dtype=np.int64)
    dp[:, 0] = cost[:, 0]

    band = 8
    for p in range(1, n_pings):
        for r in range(min_index, n_range):
            start = max(min_index, r - band)
            end = min(n_range, r + band + 1)
            prev_rows = np.arange(start, end)
            penalties = transition_penalty * np.abs(prev_rows - r)
            penalties += upward_penalty * np.maximum(prev_rows - r, 0)
            scores = dp[start:end, p - 1] + penalties
            best_local = int(np.argmin(scores))
            dp[r, p] = cost[r, p] + scores[best_local]
            back[r, p] = prev_rows[best_local]

    rows = np.full(n_pings, -1, dtype=np.int64)
    rows[-1] = int(np.argmin(dp[min_index:, -1])) + min_index
    for p in range(n_pings - 1, 0, -1):
        rows[p - 1] = back[rows[p], p]

    return range_axis[rows], rows

In [ ]:
ds = load_dataset(ZARR_PATH, ping_slice=PING_SLICE, max_range_m=MAX_RANGE_M)
sv_db = sv_db_from_dataset(ds, frequency_index=FREQUENCY_INDEX)
range_axis = ds.range.values.astype(np.float64)
ping_idx = np.arange(ds.sizes['ping_time'])
hd = heave_draft(ds, frequency_index=FREQUENCY_INDEX)

frequency_hz = float(ds.frequency.values[FREQUENCY_INDEX])
print(f'Loaded {ZARR_PATH.name}')
print(f'Pings: {ds.sizes["ping_time"]:,}  Range samples: {ds.sizes["range"]:,}  Frequency: {frequency_hz:.0f} Hz')

bottom_simple = simple_bottom_detector.detect_bottom(ds, params)
bottom_edge = edge_bottom_detector.detect_bottom(ds, params)
bottom_angle = angle_bottom_detector.detect_bottom(ds, params)
bottom_combined = combined_bottom_detector.detect_bottom(ds, params)
bottom_dp = dp_offline_bottom_detector.detect_bottom(ds, params)

range_simple = depth_to_range(bottom_simple.values, hd, params.offset)
range_edge = depth_to_range(bottom_edge.values, hd, params.offset)
range_angle = depth_to_range(bottom_angle.values, hd, params.offset)
range_combined = depth_to_range(bottom_combined.values, hd, params.offset)
range_dp = range_axis[bottom_dp.bottom_index.values]

range_gradient = np_bottom_detection.detect_bottom_gradient(
    sv_db,
    range_axis,
    minimum_range=params.minimum_range,
    gradient_window=5,
    offset=params.offset,
)

range_ensemble = np_bottom_detection.ensemble_bottom(
    [range_simple, range_edge, range_gradient],
    max_deviation=5.0,
    primary_count=2,
)
range_ensemble_pp, ensemble_outliers = np_bottom_detection.post_process_bottom(
    range_ensemble,
    max_deviation_m=10.0,
    window=31,
    fill_gaps=True,
)

range_seam, seam_idx = detect_bottom_seam(
    sv_db,
    range_axis,
    minimum_range=params.minimum_range,
    transition_penalty=0.8,
    upward_penalty=0.8,
)

depth_gradient = range_to_depth(range_gradient, hd, params.offset)
depth_ensemble_pp = range_to_depth(range_ensemble_pp, hd, params.offset)
depth_seam = range_to_depth(range_seam, hd, params.offset)

mask_combined, idx_combined = line_to_mask(range_combined, range_axis)
mask_gradient, idx_gradient = line_to_mask(range_gradient, range_axis)
mask_ensemble_pp, idx_ensemble_pp = line_to_mask(range_ensemble_pp, range_axis)
mask_seam, idx_seam = line_to_mask(range_seam, range_axis)
mask_dp = bottom_dp.bottom_range.transpose('range', 'ping_time').values.astype(np.uint8)

depths = {
    'simple': bottom_simple.values,
    'edge': bottom_edge.values,
    'angle': bottom_angle.values,
    'combined': bottom_combined.values,
    'gradient': depth_gradient,
    'ensemble_pp': depth_ensemble_pp,
    'seam': depth_seam,
    'dp_offline': bottom_dp.bottom_depth.values,
}

ranges = {
    'simple': range_simple,
    'edge': range_edge,
    'angle': range_angle,
    'combined': range_combined,
    'gradient': range_gradient,
    'ensemble_pp': range_ensemble_pp,
    'seam': range_seam,
    'dp_offline': range_dp,
}

summary = pd.DataFrame([detector_stats(name, arr) for name, arr in depths.items()]).sort_values('detector').reset_index(drop=True)
summary

In [ ]:
reference_name = 'combined'
reference = depths[reference_name]
agreement = pd.DataFrame([
    compare_to_reference(name, arr, reference)
    for name, arr in depths.items()
    if name != reference_name
]).sort_values('rmse_delta_m')
agreement

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 12), sharex=True)

im0 = axes[0].imshow(
    sv_db,
    aspect='auto',
    origin='upper',
    vmin=-80,
    vmax=-34,
    extent=[ping_idx[0], ping_idx[-1], range_axis[-1], range_axis[0]],
)
axes[0].plot(ping_idx, range_simple, color='cyan', linewidth=1.0, alpha=0.7, label='Simple')
axes[0].plot(ping_idx, range_edge, color='orange', linewidth=1.0, alpha=0.7, label='Edge')
axes[0].plot(ping_idx, range_angle, color='white', linewidth=1.0, alpha=0.7, label='Angle')
axes[0].plot(ping_idx, range_combined, color='magenta', linewidth=1.2, alpha=0.95, label='Combined')
axes[0].set_title('Legacy CRIMAC-style detectors')
axes[0].set_ylabel('Range from transducer (m)')
axes[0].legend(loc='upper right', ncol=4)

axes[1].imshow(
    sv_db,
    aspect='auto',
    origin='upper',
    vmin=-80,
    vmax=-34,
    extent=[ping_idx[0], ping_idx[-1], range_axis[-1], range_axis[0]],
)
axes[1].plot(ping_idx, range_gradient, color='tab:blue', linewidth=1.0, alpha=0.8, label='Gradient')
axes[1].plot(ping_idx, range_ensemble_pp, color='tab:red', linewidth=1.2, alpha=0.9, label='Ensemble+PP')
axes[1].plot(ping_idx, range_seam, color='gold', linewidth=1.2, alpha=0.9, label='Seam')
axes[1].plot(ping_idx, range_dp, color='lime', linewidth=1.6, alpha=0.95, label='DP offline')
axes[1].plot(ping_idx, range_combined, color='magenta', linewidth=0.8, alpha=0.5, label='Combined ref')
axes[1].set_title('Offline / labeling-oriented methods')
axes[1].set_xlabel('Ping index')
axes[1].set_ylabel('Range from transducer (m)')
axes[1].legend(loc='upper right', ncol=5)

fig.colorbar(im0, ax=axes, label='Sv (dB)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)

axes[0].plot(ping_idx, bottom_dp.bottom_confidence.values, color='tab:green', linewidth=1.2, label='DP confidence')
axes[0].axhline(params.dp_min_confidence, color='red', linestyle='--', linewidth=1.0, label='Low-confidence threshold')
axes[0].scatter(
    ping_idx[bottom_dp.bottom_low_confidence.values.astype(bool)],
    bottom_dp.bottom_confidence.values[bottom_dp.bottom_low_confidence.values.astype(bool)],
    color='red', s=12, label='Low-confidence pings'
)
axes[0].set_ylabel('Confidence')
axes[0].set_title('DP offline confidence and fallback behaviour')
axes[0].legend(loc='lower right')

axes[1].plot(ping_idx, bottom_dp.bottom_is_dropout.values, color='tab:purple', linewidth=1.0, label='DP dropout')
axes[1].scatter(ping_idx[ensemble_outliers], np.ones(ensemble_outliers.sum()), color='tab:red', s=8, label='Ensemble+PP outliers')
axes[1].set_ylabel('Flags')
axes[1].legend(loc='upper right')

axes[2].plot(ping_idx, depths['dp_offline'] - depths['combined'], color='lime', linewidth=1.1, label='DP - Combined')
axes[2].plot(ping_idx, depths['seam'] - depths['combined'], color='gold', linewidth=1.0, label='Seam - Combined')
axes[2].plot(ping_idx, depths['ensemble_pp'] - depths['combined'], color='tab:red', linewidth=1.0, label='Ensemble+PP - Combined')
axes[2].axhline(0.0, color='black', linestyle='--', linewidth=0.8)
axes[2].set_ylabel('Depth delta (m)')
axes[2].set_xlabel('Ping index')
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True, sharey=True)

plots = [
    ('Combined mask', mask_combined, 'gray', 0, 1),
    ('DP offline mask', mask_dp, 'gray', 0, 1),
    ('Seam mask', mask_seam, 'gray', 0, 1),
    ('Gradient mask', mask_gradient, 'gray', 0, 1),
    ('Ensemble+PP mask', mask_ensemble_pp, 'gray', 0, 1),
    ('DP - Combined', mask_dp.astype(np.int16) - mask_combined.astype(np.int16), 'coolwarm', -1, 1),
]

for ax, (title, image, cmap, vmin, vmax) in zip(axes.flat, plots):
    im = ax.imshow(
        image,
        aspect='auto',
        origin='upper',
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extent=[ping_idx[0], ping_idx[-1], range_axis[-1], range_axis[0]],
    )
    ax.set_title(title)
    ax.set_xlabel('Ping index')
    ax.set_ylabel('Range (m)')
    fig.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
comparison_table = summary.merge(agreement, on='detector', how='left')
comparison_table.sort_values(['rmse_delta_m', 'median_ping_to_ping_absdiff_m'], na_position='last').reset_index(drop=True)